# Abgabe 1 – Kapitel 2: End-to-End Machine-Learning-Projekt

Programmieraufgaben aus Géron Kapitel 2 (California Housing). Die Pipeline wächst schrittweise, der RMSE wird nach jeder Stufe gemessen.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import tarfile
import urllib.request

Daten laden.

In [ ]:
def load_housing_data():
    tarball_path = Path('datasets/housing.tgz')
    if not tarball_path.is_file():
        Path('datasets').mkdir(parents=True, exist_ok=True)
        url = 'https://github.com/ageron/data/raw/main/housing.tgz'
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path='datasets')
    return pd.read_csv(Path('datasets/housing/housing.csv'))

housing = load_housing_data()
housing.head()

Ergebnis: acht numerische Features und die kategoriale Spalte ocean_proximity.

Stratifizierter Train/Test-Split nach Einkommen.

In [ ]:
from sklearn.model_selection import train_test_split

housing['income_cat'] = pd.cut(housing['median_income'],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

train_set, test_set = train_test_split(
    housing, test_size=0.2, stratify=housing['income_cat'], random_state=42)

for subset in (train_set, test_set):
    subset.drop('income_cat', axis=1, inplace=True)

Features und Labels trennen.

In [ ]:
housing = train_set.drop('median_house_value', axis=1)
housing_labels = train_set['median_house_value'].copy()
print('Predictors:', housing.shape, '| Labels:', housing_labels.shape)

Ergebnis: rund 16500 Distrikte im Trainingsset mit neun Feature-Spalten.

Basis-Vorverarbeitung: Imputer und Skalierung für Zahlen, One-Hot für die Kategorie.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

num_pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
cat_pipeline = make_pipeline(SimpleImputer(strategy='most_frequent'),
                             OneHotEncoder(handle_unknown='ignore'))

preprocessing = ColumnTransformer([
    ('num', num_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', cat_pipeline, make_column_selector(dtype_include=object)),
])

print('Basis-Vorverarbeitung bereit.')

## Stufe 1: SVR mit GridSearchCV

SVR mit Gitter über kernel, C und gamma; 3-fach-CV auf den ersten 5000 Instanzen.

In [ ]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

svr_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('svr', SVR()),
])

param_grid = [
    {'svr__kernel': ['linear'], 'svr__C': [10., 100., 1000., 10000.]},
    {'svr__kernel': ['rbf'], 'svr__C': [1.0, 10., 100., 1000.],
     'svr__gamma': [0.01, 0.1, 1.0]},
]

grid_search = GridSearchCV(svr_pipeline, param_grid, cv=3,
                           scoring='neg_root_mean_squared_error')
grid_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

rmse_1 = -grid_search.best_score_
print('Beste Parameter:', grid_search.best_params_)
print('Stufe 1 RMSE: {:.0f}'.format(rmse_1))

Ergebnis: RMSE rund 69500, etwa auf dem Niveau einer linearen Regression. Das beste C liegt am oberen Gitterrand.

## Stufe 2: RandomizedSearchCV

Gleiche Pipeline, Hyperparameter aus Verteilungen mit C bis 200000.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, expon

param_distribs = {
    'svr__kernel': ['linear', 'rbf'],
    'svr__C': loguniform(20, 200_000),
    'svr__gamma': expon(scale=1.0),
}

rnd_search = RandomizedSearchCV(
    svr_pipeline, param_distributions=param_distribs, n_iter=50, cv=3,
    scoring='neg_root_mean_squared_error', random_state=42)
rnd_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

best = rnd_search.best_params_
rmse_2 = -rnd_search.best_score_
print('Beste Parameter:', best)
print('Stufe 2 RMSE: {:.0f}'.format(rmse_2))

Ergebnis: RMSE fällt auf rund 57300; gefunden wird ein großes C um 157000 mit rbf-Kernel.

## Stufe 3: SelectFromModel

Feature-Auswahl per Random Forest vor dem SVR.

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

svr_best = SVR(kernel=best['svr__kernel'], C=best['svr__C'],
               gamma=best.get('svr__gamma', 'scale'))

selector_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('selector', SelectFromModel(
        RandomForestRegressor(n_estimators=100, random_state=42), threshold=0.005)),
    ('svr', svr_best),
])

rmse_3 = -cross_val_score(selector_pipeline, housing.iloc[:5000], housing_labels.iloc[:5000],
                          cv=3, scoring='neg_root_mean_squared_error').mean()
selector_pipeline.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

print('Stufe 2 (ohne Auswahl) RMSE: {:.0f}'.format(rmse_2))
print('Stufe 3 (mit Auswahl)  RMSE: {:.0f}'.format(rmse_3))

Ergebnis: RMSE praktisch unverändert (~57400), aber mit weniger Features.

In [ ]:
feature_names = selector_pipeline.named_steps['preprocessing'].get_feature_names_out()
selector = selector_pipeline.named_steps['selector']
importances = selector.estimator_.feature_importances_
support = selector.get_support()

order = np.argsort(importances)[::-1]
print('{:12s} {:35s} {}'.format('Status', 'Feature', 'Wichtigkeit'))
print('-' * 60)
for i in order:
    status = 'behalten' if support[i] else 'verworfen'
    print('{:12s} {:35s} {:.4f}'.format(status, feature_names[i], importances[i]))
print('\nBehalten:', support.sum(), 'von', len(support), 'Features')

Ergebnis: median_income ist das wichtigste Feature; nur wenige unwichtige fallen weg.

## Stufe 4: KNN-Geo-Merkmal

KNN-Schätzung des Nachbarschaftspreises aus Länge und Breite, skaliert, als zusätzliches Feature.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.neighbors import KNeighborsRegressor
from sklearn.utils.validation import check_is_fitted

class FeatureFromRegressor(BaseEstimator, TransformerMixin):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y=None):
        estimator_ = clone(self.estimator)
        estimator_.fit(X, y)
        self.estimator_ = estimator_
        self.n_features_in_ = self.estimator_.n_features_in_
        if hasattr(self.estimator, 'feature_names_in_'):
            self.feature_names_in_ = self.estimator.feature_names_in_
        return self

    def transform(self, X):
        check_is_fitted(self)
        predictions = self.estimator_.predict(X)
        if predictions.ndim == 1:
            predictions = predictions.reshape(-1, 1)
        return predictions

    def get_feature_names_out(self, names=None):
        check_is_fitted(self)
        n_outputs = getattr(self.estimator_, 'n_outputs_', 1)
        prefix = self.estimator_.__class__.__name__.lower()
        return [f'{prefix}_prediction_{i}' for i in range(n_outputs)]

In [ ]:
# Geo-Merkmal wird skaliert, sonst dominiert sein grosser Wert die SVR-Distanzen.
geo_pipeline = make_pipeline(
    FeatureFromRegressor(KNeighborsRegressor(n_neighbors=5)),
    StandardScaler())

preprocessing_geo = ColumnTransformer([
    ('geo', geo_pipeline, ['longitude', 'latitude']),
    ('num', num_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', cat_pipeline, make_column_selector(dtype_include=object)),
])

full_pipeline = Pipeline([
    ('preprocessing', preprocessing_geo),
    ('selector', SelectFromModel(
        RandomForestRegressor(n_estimators=100, random_state=42), threshold=0.005)),
    ('svr', svr_best),
])

rmse_4 = -cross_val_score(full_pipeline, housing.iloc[:5000], housing_labels.iloc[:5000],
                          cv=3, scoring='neg_root_mean_squared_error').mean()
print('Stufe 3 (ohne Geo-Merkmal) RMSE: {:.0f}'.format(rmse_3))
print('Stufe 4 (mit Geo-Merkmal)  RMSE: {:.0f}'.format(rmse_4))

Ergebnis: RMSE sinkt deutlich auf rund 49800; das Lage-Merkmal ist sehr informativ.

## Stufe 5: Vorverarbeitung tunen

GridSearch über n_neighbors des Geo-Merkmals und die Imputer-Strategie.

In [ ]:
param_grid_pre = {
    'preprocessing__geo__featurefromregressor__estimator__n_neighbors': [3, 5, 7],
    'preprocessing__num__simpleimputer__strategy': ['mean', 'median'],
}

grid_pre = GridSearchCV(full_pipeline, param_grid_pre, cv=3,
                        scoring='neg_root_mean_squared_error')
grid_pre.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

rmse_5 = -grid_pre.best_score_
print('Beste Vorverarbeitungs-Optionen:', grid_pre.best_params_)
print('Stufe 5 RMSE: {:.0f}'.format(rmse_5))

Ergebnis: kleine weitere Verbesserung auf rund 49000.

## Aufgabe 6: StandardScalerClone

Eigene StandardScaler-Implementierung mit inverse_transform und Feature-Namen.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):
        self.with_mean = with_mean

    def fit(self, X, y=None):
        X_orig = X
        X = check_array(X)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.scale_[self.scale_ == 0] = 1
        self.n_features_in_ = X.shape[1]
        if hasattr(X_orig, 'columns'):
            self.feature_names_in_ = np.array(X_orig.columns, dtype=object)
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('Falsche Anzahl Features')
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

    def inverse_transform(self, X):
        check_is_fitted(self)
        X = check_array(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('Falsche Anzahl Features')
        X = X * self.scale_
        return X + self.mean_ if self.with_mean else X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return getattr(self, 'feature_names_in_',
                           np.array([f'x{i}' for i in range(self.n_features_in_)], dtype=object))
        if len(input_features) != self.n_features_in_:
            raise ValueError('Falsche Anzahl input_features')
        if hasattr(self, 'feature_names_in_') and not np.all(self.feature_names_in_ == input_features):
            raise ValueError('input_features passt nicht zu feature_names_in_')
        return np.asarray(input_features, dtype=object)

In [ ]:
np.random.seed(42)
X_demo = np.random.rand(20, 3)
scaler = StandardScalerClone()
X_back = scaler.inverse_transform(scaler.fit_transform(X_demo))
print('inverse_transform ~ Original:', np.allclose(X_demo, X_back))

df_demo = pd.DataFrame(X_demo, columns=['a', 'b', 'c'])
scaler.fit(df_demo)
print('feature_names_in_:', scaler.feature_names_in_)
print('get_feature_names_out():', scaler.get_feature_names_out())

scaler.fit(X_demo)
print('ohne Namen:', scaler.get_feature_names_out())

Ergebnis: inverse_transform rekonstruiert das Original; die Feature-Namen werden korrekt zurückgegeben.

## Zusammenfassung

RMSE nach jeder Stufe.

In [ ]:
print('Entwicklung des RMSE ueber die wachsende Pipeline (kleiner = besser):')
print('-' * 50)
print(f'  Stufe 1  SVR + GridSearchCV          {rmse_1:>8.0f}')
print(f'  Stufe 2  + RandomizedSearchCV        {rmse_2:>8.0f}')
print(f'  Stufe 3  + SelectFromModel           {rmse_3:>8.0f}')
print(f'  Stufe 4  + KNN-Geo-Merkmal           {rmse_4:>8.0f}')
print(f'  Stufe 5  + Vorverarbeitung getunt    {rmse_5:>8.0f}')

Ergebnis: der RMSE fällt von rund 69500 auf rund 49000; die größten Schritte bringen die Zufallssuche und das Geo-Merkmal.